In [ ]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [ ]:
%run ../common/enviroment_config

In [ ]:
%run ../common/silver_helpers

In [ ]:
from pyspark.sql import Functions as F
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [ ]:
circuits_df = (
    spark.table(bronze_table).filter((F.col("batch_id")== v_batch_id ))
)
display(circuits_df)

In [ ]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [ ]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)
display(circuits_renamed_df)

In [ ]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)
display(circuits_valid_df)

In [ ]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])
display(circuits_distinct_df)

In [ ]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)
display(circuits_final_df)

In [ ]:
write_to_silver(
    input_df=circuits_final_df,
    target_table=silver_table,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"        
    ]
)
display(spark.table(silver_table))